# Stage 3: DPO Preference Alignment — Healthcare FAQ Assistant

In [3]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-_rs5ic8y/unsloth_7613d2b66a3e436ebeae519f6bb062df
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-_rs5ic8y/unsloth_7613d2b66a3e436ebeae519f6bb062df
  Resolved https://github.com/unslothai/unsloth.git to commit 5211b506e1d59d647528631e03cef1cb43ccbf7d
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 53.9 MB/s eta 0:00:00


## 1. Load Preference Dataset

In [5]:
import json
from datasets import Dataset

with open("/content/sample_data/preference_dataset.jsonl", "r", encoding="utf-8") as f:
    pref_data = [json.loads(line) for line in f if line.strip()]

print(f"Total preference examples: {len(pref_data)}")
print("\nSample:")
print(json.dumps(pref_data[0], indent=2))

dataset = Dataset.from_list(pref_data)
print(f"\nDataset: {dataset}")

Total preference examples: 54

Sample:
{
  "prompt": "What are the symptoms of a heart attack?",
  "chosen": "Heart attack symptoms include chest pain or pressure, pain radiating to the arm, neck, or jaw, shortness of breath, cold sweats, nausea, and lightheadedness. Women may experience atypical symptoms such as unusual fatigue and back pain. If you suspect a heart attack, call emergency services (911) immediately \u2014 every minute counts.",
  "rejected": "Just take some painkillers and lie down. It's probably just indigestion."
}

Dataset: Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 54
})


## 2. Load SFT Model

In [9]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 512
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/tinyllama-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Load the SFT adapter weights on top

model, tokenizer = FastLanguageModel.from_pretrained(
    "Deeptid123/HealthAiAsst-sft-model",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)


print("SFT adapter loaded successfully.")

==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Unsloth: Will load unsloth/tinyllama-bnb-4bit as a legacy tokenizer.


==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/919 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.62M [00:00<?, ?B/s]

Unsloth: Will load Deeptid123/HealthAiAsst-sft-model as a legacy tokenizer.


SFT adapter loaded successfully.


## 3. Apply LoRA for DPO Training

In [10]:
from unsloth import FastLanguageModel

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)
model.print_trainable_parameters()

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.6.9 patched 22 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


## 4. Format DPO Dataset

In [11]:
alpaca_prompt = """Below is a question about health and medicine. Write a clear, accurate, and helpful response.

### Question:
{}

### Answer:
{}"""

def format_dpo(examples):
    prompts = [alpaca_prompt.format(p, "") for p in examples["prompt"]]
    chosens = [c + tokenizer.eos_token for c in examples["chosen"]]
    rejecteds = [r + tokenizer.eos_token for r in examples["rejected"]]
    return {
        "prompt": prompts,
        "chosen": chosens,
        "rejected": rejecteds,
    }

dpo_dataset = dataset.map(format_dpo, batched=True)
print(f"DPO dataset: {dpo_dataset}")
print("\nSample prompt:")
print(dpo_dataset[0]["prompt"][:300])

Map:   0%|          | 0/54 [00:00<?, ? examples/s]

DPO dataset: Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 54
})

Sample prompt:
Below is a question about health and medicine. Write a clear, accurate, and helpful response.

### Question:
What are the symptoms of a heart attack?

### Answer:



## 5. Configure and Run DPO Training

In [12]:
from trl import DPOTrainer, DPOConfig
from unsloth import is_bfloat16_supported

dpo_config = DPOConfig(
    beta = 0.1,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    num_train_epochs = 2,
    learning_rate = 5e-5,
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    logging_steps = 10,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 42,
    output_dir = "outputs_dpo",
    report_to = "none",
    max_prompt_length = 256,
    max_length = 512,
)

dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None,  # use implicit reference (PEFT approach)
    args = dpo_config,
    train_dataset = dpo_dataset,
    tokenizer = tokenizer,
)

print("Starting DPO alignment...")
dpo_stats = dpo_trainer.train()
print(f"\nDPO training complete! Final loss: {dpo_stats.training_loss:.4f}")

Extracting prompt in train dataset (num_proc=6):   0%|          | 0/54 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=6):   0%|          | 0/54 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=6):   0%|          | 0/54 [00:00<?, ? examples/s]

Starting DPO alignment...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 54 | Num Epochs = 2 | Total steps = 14
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 12,615,680 of 1,112,664,064 (1.13% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


...

Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected
10,0.467221,0.527730,-0.101181,0.794872,0.628912,-235.807098,-80.629349,-3.775740,-5.385036


Unsloth: Restored added_tokens_decoder metadata in outputs_dpo/checkpoint-14/tokenizer_config.json.


tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs_dpo/checkpoint-14.



DPO training complete! Final loss: 0.3757


## 6. Save the DPO-Aligned Model

In [13]:
model.save_pretrained("../models/dpo_adapter")
tokenizer.save_pretrained("../models/dpo_adapter")
print("DPO-aligned model saved to ../models/dpo_adapter")

Unsloth: Restored added_tokens_decoder metadata in ../models/dpo_adapter/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in ../models/dpo_adapter.


DPO-aligned model saved to ../models/dpo_adapter


## 7. Test the DPO-Aligned Model

In [14]:
FastLanguageModel.for_inference(model)

def generate_answer_dpo(question, max_new_tokens=200):
    prompt = alpaca_prompt.format(question, "")
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens = max_new_tokens,
        temperature = 0.7,
        top_p = 0.9,
        do_sample = True,
        pad_token_id = tokenizer.eos_token_id,
    )
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = full_output.split("### Answer:")[-1].strip()
    return answer

In [15]:
test_questions = [
    "What are the symptoms of type 2 diabetes?",
    "How can I lower my blood pressure naturally?",
    "What should I do during a heart attack?",
    "How much exercise should adults get per week?",
    "What is the difference between a cold and the flu?",
    "What foods should I eat for a healthy heart?",
    "How can I improve my sleep quality?",
    "What are the warning signs of a stroke?",
    "How can I prevent kidney stones?",
    "What is the recommended daily water intake?",
]

print("=" * 70)
print("DPO-Aligned Model Responses")
print("=" * 70)
for i, q in enumerate(test_questions, 1):
    print(f"\nQ{i}: {q}")
    print(f"A: {generate_answer_dpo(q)}")
    print("-" * 50)

Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


DPO-Aligned Model Responses

Q1: What are the symptoms of type 2 diabetes?


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

A: Type 1 diabetes is a disease in which the pancreas produces little or no insulin. The pancreas is the gland in the abdomen that produces the hormone insulin. In type 1 diabetes, the pancreas is not able to make enough insulin. As a result, the body
--------------------------------------------------

Q2: How can I lower my blood pressure naturally?


Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

A: Sleep apnea is a sleep disorder in which the airway in the throat or tongue collapses during sleep, causing snoring, restless sleep, and interruptions in the sleep
--------------------------------------------------

Q8: What are the warning signs of a stroke?


Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: **Warning signs of stroke:**

  * **Shortness of breath:** Shortness of breath is a sign of a heart condition.
  * **Sudden weakness or numbness:** Sudden weakness or numbness is a sign of a heart condition.
  * **Chest pain:** Chest pain is a sign of a heart condition.
  * **Nausea or vomiting:** Nausea or vomiting is a sign of a heart condition.
  * **Vision changes:** Vision changes are a sign of a heart condition.
  * **Pain in the jaw, face, or throat:** Pain in the jaw, face, or throat is a sign of a heart condition.
  * **Fainting or lightheadedness:** Fainting or lightheadedness is a sign of a heart condition.
  * **
--------------------------------------------------

Q9: How can I prevent kidney stones?


Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: Kidney stones are small, hard masses of minerals and salts that form in the kidneys. They are a common problem, especially in people over age 50.

It is not clear exactly how kidney stones form. However, there are several risk factors that increase the chances that a person will develop a kidney stone:

  * Having a family history of kidney stones.
  * Having a family history of kidney stones and having had a kidney stone in the past.
  * Being obese.
  * Being female.
  * Having a history of kidney stones.

Kidney stones are also associated with a number of other risk factors:

  * Having diabetes.
  * Having high blood pressure.
  * Having high cholesterol.
  * Having high uric acid levels.
  * Having high calcium levels.

If you
--------------------------------------------------

Q10: What is the recommended daily water intake?
A: The recommended daily water intake for children is 1.5 L. Children need 1.5 L of water per day to stay hydrated. Children
-----------------------------

In [16]:
import shutil, os

os.makedirs("/content/drive/MyDrive/healthcare_models",exist_ok=True)
shutil.copytree("../models/dpo_adapter","/content/drive/MyDrive/healthcare_models/dpo_adapter")
print("Model copied to Google Drive")

Model copied to Google Drive


In [21]:
ALPACA_PROMPT = """Below is a question about health and medicine. Write a clear, accurate, and helpful response.

### Question:
{}

### Answer:
"""

def generate_answer(question: str, max_new_tokens: int = 256) -> str:
    prompt = ALPACA_PROMPT.format(question)

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "### Answer:" in decoded:
        answer = decoded.split("### Answer:")[-1].strip()
    else:
        answer = decoded.replace(prompt, "").strip()

    return answer


# test
question = "What are the symptoms of type 2 diabetes?"
print(generate_answer(question))

Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The symptoms of type 2 diabetes are similar to those of type 1, but they include:

  * Unusual thirst
  * Weight loss
  * Blurred vision
  * Increased hunger
  * Tiredness
  * Dry mouth
  * Dry skin
  * Urine that is dark in color
  * Feeling of a "full" feeling
  * Sweating
  * Tiredness
  * Unusual fatigue
  * Changes in the way your urine smells
  * Muscle pain or weakness
  * Fatigue
  * Slowed or troubled breathing
  * Chest pain
  * Swelling in the ankles, feet, or legs
  * Tingling or numbness in the arms, legs, fingers, toes, or feet
  * Blurred vision
  * Blurred or double vision
  * Red, swollen, or bluish skin in the groin area
  * Rashes or hives
  * Dark urine
  * Weight gain
  * Tired


In [26]:

from google.colab import userdata
userdata.get('HF_TOKEN_WRITE')


'[REDACTED]'

In [28]:
### pushing to HF

# =============================
# 1. Install + imports
# =============================
!pip install -q huggingface_hub

from huggingface_hub import HfApi
from google.colab import userdata

# =============================
# 2. (OPTIONAL) get HF token from environment
# =============================
# Works if you stored token in environment / Colab secrets / .env
HF_TOKEN = userdata.get("HF_TOKEN_WRITE")  # make sure your secret name matches

# =============================
# 3. Save model locally
# =============================
SAVE_PATH = "final_model"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("✅ Model saved locally at:", SAVE_PATH)

# =============================
# 4. Upload to Hugging Face
# =============================
REPO_ID = "Deeptid123/HealthAiAsst-dpo-model"

api = HfApi()

api.create_repo(
    repo_id=REPO_ID,
    repo_type="model",
    exist_ok=True,
    token=HF_TOKEN,
)

api.upload_folder(
    folder_path=SAVE_PATH,
    repo_id=REPO_ID,
    repo_type="model",
    token=HF_TOKEN,
)

print("🚀 Model successfully pushed to HF:", REPO_ID)

# =============================
# 5. TEST LOAD FROM HF (IMPORTANT)
# =============================
from unsloth import FastLanguageModel

test_model, test_tokenizer = FastLanguageModel.from_pretrained(
    REPO_ID,
    max_seq_length=512,
    load_in_4bit=True,
)

print("✅ Model reloaded from HF successfully!")

# =============================
# 6. TEST INFERENCE
# =============================

ALPACA_PROMPT = """Below is a question about health and medicine. Write a clear, accurate, and helpful response.

### Question:
{}

### Answer:
"""

def generate_answer(question: str):
    prompt = ALPACA_PROMPT.format(question)
    inputs = test_tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = test_model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=test_tokenizer.eos_token_id,
    )

    decoded = test_tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "### Answer:" in decoded:
        return decoded.split("### Answer:")[-1].strip()
    return decoded.replace(prompt, "").strip()

# quick test
question = "What are the symptoms of type 2 diabetes?"
print("\nQ:", question)
print("A:", generate_answer(question))

✅ Model saved locally at: final_model
🚀 Model successfully pushed to HF: Deeptid123/HealthAiAsst-dpo-model
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Unsloth: Will load Deeptid123/HealthAiAsst-sft-model as a legacy tokenizer.


adapter_model.safetensors:   0%|          | 0.00/50.5M [00:00<?, ?B/s]

Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ Model reloaded from HF successfully!

Q: What are the symptoms of type 2 diabetes?


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


A: Diabetes mellitus type 2 is a chronic metabolic disease. It is characterized by high blood glucose levels, excessive production of insulin, and impaired glucose tolerance. Diabetes mellitus type 2 is the most common form of diabetes and is the result of a combination of genetic and environmental factors. In people with diabetes mellitus type 2, the pancreas produces less insulin than normal, resulting in high blood glucose levels. This condition is often associated with obesity, sedentary lifestyle, and elevated blood pressure. The onset of diabetes mellitus type 2 usually occurs in adulthood, but the disease can develop earlier in life, especially in children and young adults.

Diabetes mellitus type 2 is characterized by high blood glucose levels, excessive production of insulin, and impaired glucose tolerance. These symptoms can affect any part of the body, including the eyes, kidneys, heart, nerves, and feet.

Diabetes mellitus


## Summary

DPO alignment improves over the SFT model by learning from preference data — training the model to prefer correct, safe, domain-specific answers over incorrect or harmful ones. The final DPO-aligned model should give more professional, accurate, and helpful healthcare responses.